# Caderno de análise — PQC no ATmega4809 (CMP223)

Caderno de programação literal: cada etapa, do dado bruto à conclusão, fica
registrada aqui e pode ser re-executada (`jupyter nbconvert --execute`).

**Fonte dos dados:** somente observações de **bancada** (`data_source = bench`),
identificadas por `src/analysis/catalog.py` a partir da presença de
`raw/telemetry_*.json` (arquivo gravado apenas pelo `trigger.py` com a placa
conectada). Pastas simuladas, vazias ou legadas são **excluídas**.

**Limitações conhecidas (antes de olhar os números):**
1. *Latência* = tempo medido no host do envio do comando até o 1º byte de
   resposta (`t_exec_ms`). Inclui overhead fixo de USB/UART; o analisador lógico
   (largura do pulso GPIO) refinará isso.
2. *Energia* = **não medida** (shunt e bateria ainda não instalados). Nenhum
   valor de energia é reportado neste caderno.
3. *Clock*: apenas 20 MHz foi coletado na bancada até agora.
4. *Ordem*: na bancada, cada configuração rodou em bloco (ciclo fixo K→E→D→M),
   não na ordem aleatória do `schedule.json`.

In [1]:
import glob, json, os, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotnine as p9
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

# Raiz do repositório (o caderno roda a partir de src/analysis/).
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EXP = ROOT / "data" / "experiments"
OUT = ROOT / "data" / "processed"
FIG = ROOT / "artigo" / "figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

import subprocess
from matplotlib import font_manager

# Figuras no padrão do artigo (acmsmall): largura de texto = 395,8 pt = 5,5 in,
# fonte Libertinus Serif (sucessora da Linux Libertine do corpo; o OTF da
# Libertine tem métricas verticais quebradas no matplotlib), 8 pt. A fonte é localizada pela
# distribuição TeX (kpsewhich); sem ela, cai para serif genérica.
TEXTW = 395.8225 / 72.27
for f in ("LibertinusSerif-Regular.otf", "LibertinusSerif-Bold.otf", "LibertinusSerif-Italic.otf"):
    try:
        path = subprocess.run(["kpsewhich", f], capture_output=True, text=True).stdout.strip()
        if path:
            font_manager.fontManager.addfont(path)
    except FileNotFoundError:
        pass
sns.set_theme(style="ticks", context="paper")
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Libertinus Serif", "DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.6,
    "grid.linewidth": 0.4, "lines.linewidth": 1.0, "axes.grid": True, "grid.alpha": 0.4,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "figure.dpi": 150,
    "axes.formatter.use_locale": False,
})
LABEL = {"kyber512": "Kyber-512", "ctru-light": "CTRU-Light", "ecdh": "ECDH"}
def br(x, nd=1):
    """Número com vírgula decimal (padrão do texto em português)."""
    return f"{x:,.{nd}f}".replace(",", "X").replace(".", ",").replace("X", ".")
PAL = {"kyber512": "#0072B2", "ctru-light": "#D55E00", "ecdh": "#009E73"}  # Okabe-Ito
OPS = {"K": "keygen", "E": "encaps", "D": "decaps", "M": "msgcrypt"}


Matplotlib created a temporary cache directory at /tmp/matplotlib-kkvppij8 because the default path (/home/eduabdala/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## 1. Proveniência: quais experimentos entram na análise

In [2]:
cat = pd.read_csv(EXP / "catalog.csv")
display(cat.groupby("source").agg(experimentos=("experiment", "count"),
                                   observacoes=("n_observations", "sum")))
bench_exps = cat.loc[cat.source == "bench", "experiment"].tolist()
bench_exps

,experimentos,observacoes
source,,
bench,10,2880
bench_legacy,4,60
empty,8,0
sim,8,3056


['exp_20261104_224548_exp0_full_raw_2241',
 'exp_20261104_233711_exp0_full_raw_3170',
 'exp_20261104_234001_exp0_full_raw_1522',
 'exp_20261104_234322_kyber512_3110',
 'exp_20261104_234555_kyber512_8736',
 'exp_20261104_235614_all_4877',
 'exp_20261105_001049_complete_simulation_matrix_7223',
 'exp_20261105_072304_full_pqc_campaign_7266',
 'exp_20261105_185346_bench_full_pqc_campaign_9755',
 'exp_20261105_191552_bench_full_pqc_campaign_8196']

Apenas `bench` entra. `bench_legacy` foi descartado porque nessas pastas só
sobrou o `runs.csv`, sem a telemetria bruta; o `runs.csv` delas também traz uma
coluna de energia calculada com corrente constante, que **não** é medição.

## 2. Carga da telemetria bruta em formato *tidy*

In [3]:
# telemetry_<alg>_<variant>[_<opt>_<clk>MHz].json ; arquivos antigos sem opt/clk
# foram coletados com -O3 @ 20 MHz (confirmado no schedule.json de cada pasta).
PAT = re.compile(r"telemetry_(?P<alg>kyber512|ctru-light|ecdh)_(?P<variant>asm|ref)"
                 r"(?:_(?P<opt>O3|Os)_(?P<clk>[\d.]+)MHz)?\.json")
rows = []
for e in bench_exps:
    for f in sorted(glob.glob(str(EXP / e / "raw" / "telemetry_*.json"))):
        m = PAT.match(os.path.basename(f)).groupdict()
        for i, r in enumerate(json.load(open(f))):
            rows.append({**r, "experiment": e, "seq": i,
                         "algorithm": m["alg"], "variant": m["variant"],
                         "optimization": "-" + (m["opt"] or "O3"),
                         "clock_mhz": float(m["clk"] or 20)})
df = pd.DataFrame(rows)
df["operation"] = df.cmd.map(OPS)
df["timestamp"] = pd.to_datetime(df.timestamp)
df = df[["experiment", "seq", "timestamp", "algorithm", "variant", "optimization",
         "clock_mhz", "operation", "t_exec_ms", "t_total_ms", "stack_peak",
         "sram_total", "payload_bytes", "kem_validated", "msg_validated", "status"]]
df.to_csv(OUT / "bench_tidy.csv", index=False)
print(len(df), "observações ->", OUT / "bench_tidy.csv")
df.head()

2880 observações -> /home/eduabdala/git/project-cspa-cmp223/data/processed/bench_tidy.csv


,experiment,seq,timestamp,algorithm,variant,optimization,clock_mhz,operation,t_exec_ms,t_total_ms,stack_peak,sram_total,payload_bytes,kem_validated,msg_validated,status
0,exp_20261104_224548_exp0_full_raw_2241,0,2026-11-04 22:46:13.906792,kyber512,asm,-O3,20.0,keygen,261.203,331.226,935,6144,800,None,None,OK
1,exp_20261104_224548_exp0_full_raw_2241,1,2026-11-04 22:46:14.377721,kyber512,asm,-O3,20.0,encaps,343.230,413.192,1008,6144,800,None,None,OK
2,exp_20261104_224548_exp0_full_raw_2241,2,2026-11-04 22:46:14.747926,kyber512,asm,-O3,20.0,decaps,316.649,319.974,1019,6144,32,True,None,OK
3,exp_20261104_224548_exp0_full_raw_2241,3,2026-11-04 22:46:14.839977,kyber512,asm,-O3,20.0,msgcrypt,38.368,41.244,68,6144,18,None,True,OK
4,exp_20261104_224548_exp0_full_raw_2241,4,2026-11-04 22:46:15.228782,kyber512,asm,-O3,20.0,keygen,261.436,331.200,935,6144,800,None,None,OK


## 3. Sanidade dos dados (controle experimental)

**Ordem de execução.** O `schedule.json` de cada pasta traz uma ordem
aleatória, mas o modo bench do `run_experiment.py` não a aplica: cada
configuração roda em bloco, com o ciclo fixo K→E→D→M. Por isso o gráfico de
deriva abaixo é a principal evidência de que não há efeito temporal.

In [4]:
print("status:", df.status.value_counts().to_dict())
print("kem_validated (decaps):", df.loc[df.operation == "decaps", "kem_validated"].value_counts(dropna=False).to_dict())
print("msg_validated (msgcrypt):", df.loc[df.operation == "msgcrypt", "msg_validated"].value_counts(dropna=False).to_dict())
# O pico de pilha é determinístico por configuração? (esperado: sim)
sp = df.groupby(["algorithm", "variant", "optimization", "operation"]).stack_peak.nunique()
print("configurações com stack_peak variável:", int((sp > 1).sum()))
df.groupby(["algorithm", "variant", "optimization"]).size().rename("n").to_frame()

status: {'OK': 2880}
kem_validated (decaps): {True: 707, None: 13}
msg_validated (msgcrypt): {True: 720}
configurações com stack_peak variável: 5


n
algorithm  variant optimization     
ctru-light asm     -O3           240
                   -Os           240
           ref     -O3           120
                   -Os           120
ecdh       asm     -O3            80
                   -Os            80
           ref     -O3            40
                   -Os            40
kyber512   asm     -O3           740
                   -Os           240
           ref     -O3           700
                   -Os           240

Todas as observações têm `status = OK`. O pico de pilha é constante dentro de cada
configuração, como esperado para código sem alocação dinâmica: a variabilidade
está só na latência. As configurações são **desbalanceadas** (CTRU-Light `ref` tem
menos repetições); por isso usamos ANOVA tipo II e ICs por célula.

In [5]:
# Deriva temporal ao longo da ordem de execução (Jain cap. 11): sem tendência
# esperada. Gráfico ggplot2 para kyber512 asm -O3 (maior N).
sub_drift = df[(df.algorithm == 'kyber512') & (df.variant == 'asm') & (df.optimization == '-O3')].copy()
sub_drift['seq'] = sub_drift.groupby('operation').cumcount()
p_drift = (
    p9.ggplot(sub_drift, p9.aes(x='seq', y='t_exec_ms', color='operation'))
    + p9.geom_point(size=1.2, alpha=0.75)
    + p9.labs(
        x='Ordem de execução (índice dentro da operação)',
        y='Latência (ms)',
        title='Kyber-512 asm -O3: latência por ordem de execução'
    )
    + p9.scale_color_brewer(type='qual', palette='Set1', name='Operação')
    + p9.theme_bw(base_size=8)
    + p9.theme(
        legend_position='top',
        figure_size=(7, 2.6)
    )
)
fig_drift = p_drift.draw()
fig_drift.savefig(FIG / 'drift_kyber_asm.png', dpi=200, bbox_inches='tight')
fig_drift.savefig(FIG / 'drift_kyber_asm.pdf', bbox_inches='tight')
plt.close(fig_drift)


## 4. Sumário estatístico (média, IC 95% t-Student, mediana, IQR)

In [6]:
def summarise(x):
    n = len(x); m = x.mean(); s = x.std(ddof=1)
    h = stats.t.ppf(0.975, n - 1) * s / np.sqrt(n) if n > 1 else np.nan
    q1, q3 = x.quantile([.25, .75])
    return pd.Series({"n": n, "mean_ms": m, "sd_ms": s, "ci95_ms": h,
                      "median_ms": x.median(), "iqr_ms": q3 - q1, "cv_pct": 100 * s / m})

KEYS = ["algorithm", "variant", "optimization", "operation"]
summ = df.groupby(KEYS).t_exec_ms.apply(summarise).unstack()
summ["stack_peak_b"] = df.groupby(KEYS).stack_peak.first()
summ["sram_free_pct"] = 100 * (6144 - summ.stack_peak_b) / 6144
summ["energy_uj"] = np.nan  # NA: shunt ainda não instalado
summ = summ.reset_index()
summ.to_csv(OUT / "summary_bench.csv", index=False)
summ.round(2)

,algorithm,variant,optimization,operation,n,mean_ms,sd_ms,ci95_ms,median_ms,iqr_ms,cv_pct,stack_peak_b,sram_free_pct,energy_uj
0,ctru-light,asm,-O3,decaps,60.0,151.14,0.26,0.07,151.07,0.40,0.17,3177,48.29,NaN
1,ctru-light,asm,-O3,encaps,60.0,91.33,4.21,1.09,89.17,0.58,4.61,2496,59.38,NaN
2,ctru-light,asm,-O3,keygen,60.0,91.19,5.07,1.31,95.32,10.11,5.56,2464,59.90,NaN
3,ctru-light,asm,-O3,msgcrypt,60.0,39.24,0.28,0.07,39.13,0.38,0.71,68,98.89,NaN
4,ctru-light,asm,-Os,decaps,60.0,156.72,2.29,0.59,156.27,0.29,1.46,3137,48.94,NaN
5,ctru-light,asm,-Os,encaps,60.0,93.78,4.61,1.19,91.40,0.56,4.91,2460,59.96,NaN
6,ctru-light,asm,-Os,keygen,60.0,93.62,5.46,1.41,98.08,10.92,5.84,2432,60.42,NaN
7,ctru-light,asm,-Os,msgcrypt,60.0,41.67,0.23,0.06,41.60,0.22,0.56,36,99.41,NaN
8,ctru-light,ref,-O3,decaps,30.0,272.81,0.28,0.11,272.73,0.42,0.10,3049,50.37,NaN
9,ctru-light,ref,-O3,encaps,30.0,139.94,4.11,1.53,137.94,0.61,2.94,2376,61.33,NaN


## 5. Flash ocupada (artefato de compilação, independente da bancada)

In [7]:
def hex_bytes(path):
    # Soma dos bytes dos registros de dados (tipo 00) do Intel HEX = .text + .data.
    n = 0
    for line in open(path):
        if line.startswith(":") and line[7:9] == "00":
            n += int(line[1:3], 16)
    return n

HP = re.compile(r"firmware-(?P<alg>kyber512|ctru-light|ecdh)-(?P<variant>asm|ref)-[\d.]+MHz-(?P<opt>O3|Os)\.hex")
hx = []
for f in sorted(glob.glob(str(EXP / "exp_*" / "bin" / "*.hex"))):  # ordem cronológica
    m = HP.match(os.path.basename(f))
    if m:
        hx.append({**m.groupdict(), "flash_b": hex_bytes(f), "src": Path(f).parts[-3]})
flash = (pd.DataFrame(hx).groupby(["alg", "variant", "opt"]).last().reset_index()
         .rename(columns={"alg": "algorithm", "opt": "optimization"}))
flash["optimization"] = "-" + flash.optimization
flash["flash_free_pct"] = 100 * (49152 - flash.flash_b) / 49152
flash.to_csv(OUT / "flash_sizes.csv", index=False)
flash

,algorithm,variant,optimization,flash_b,src,flash_free_pct
0,ctru-light,asm,-O3,28794,exp_20261107_181223_bench_full_la_campaign_8412,41.418457
1,ctru-light,asm,-Os,23116,exp_20261107_161316_bench_full_pqc_campaign_9549,52.970378
2,ctru-light,ref,-O3,47140,exp_20261107_181223_bench_full_la_campaign_8412,4.093424
3,ctru-light,ref,-Os,40938,exp_20261107_161316_bench_full_pqc_campaign_9549,16.711426
4,ecdh,asm,-O3,14806,exp_20261107_181223_bench_full_la_campaign_8412,69.877116
5,ecdh,asm,-Os,9622,exp_20261107_161316_bench_full_pqc_campaign_9549,80.423991
6,ecdh,ref,-O3,38752,exp_20261107_181223_bench_full_la_campaign_8412,21.158854
7,ecdh,ref,-Os,10792,exp_20261107_161316_bench_full_pqc_campaign_9549,78.043620
8,kyber512,asm,-O3,26256,exp_20261107_181223_bench_full_la_campaign_8412,46.582031
9,kyber512,asm,-Os,15414,exp_20261107_161316_bench_full_pqc_campaign_9549,68.640137


O ECDH (micro-ecc, curva secp256r1) entra na bancada a partir da campanha
`exp_20261105_191552_bench_full_pqc_campaign_8196`, a primeira a cobrir as 12
configurações. Usamos o `.hex` mais recente de cada configuração; não
verificamos se todos vieram do mesmo commit do firmware.

In [8]:
fl = flash.copy()
fl['alg_name'] = fl.algorithm.map(LABEL)
fl['cfg'] = [f"{a}\n{v}" for a, v in zip(fl.alg_name, fl.variant)]
order = ['Kyber-512\nasm', 'Kyber-512\nref', 'CTRU-Light\nasm', 'CTRU-Light\nref', 'ECDH\nasm', 'ECDH\nref']
fl['cfg'] = pd.Categorical(fl.cfg, categories=order, ordered=True)
fl['flash_kb'] = fl.flash_b / 1024.0
opt_map = {'-O3': 'Otimização -O3', '-Os': 'Otimização -Os'}
fl['opt_label'] = pd.Categorical(fl.optimization.map(opt_map), categories=['Otimização -O3', 'Otimização -Os'], ordered=True)
colors = {'Otimização -O3': '#0072B2', 'Otimização -Os': '#E69F00'}
p_flash = (
    p9.ggplot(fl, p9.aes(x='cfg', y='flash_kb', fill='opt_label'))
    + p9.geom_col(position=p9.position_dodge(width=0.65), width=0.58, color='black', size=0.4)
    + p9.geom_hline(yintercept=48, linetype='dashed', color='#B22222', size=0.8)
    + p9.annotate('text', x=5.4, y=50.2, label='Capacidade do MCU (48 KB)',
                  color='#B22222', size=7.5, fontweight='bold', ha='right', va='bottom')
    + p9.scale_fill_manual(values=colors, name='')
    + p9.scale_y_continuous(limits=(0, 56), breaks=[0, 8, 16, 24, 32, 40, 48], expand=(0, 0, 0.05, 0))
    + p9.labs(x='', y='Flash ocupada (KB)')
    + p9.theme_bw(base_size=8)
    + p9.theme(
        legend_position='top',
        legend_direction='horizontal',
        legend_title=p9.element_blank(),
        legend_box_spacing=0.05,
        panel_grid_minor=p9.element_blank(),
        panel_grid_major_x=p9.element_blank(),
        panel_grid_major_y=p9.element_line(color='#e0e0e0', size=0.4, linetype='dotted'),
        figure_size=(TEXTW, 2.3)
    )
)
fig_flash = p_flash.draw()
fig_flash.savefig(FIG / 'flash.png', dpi=300, bbox_inches='tight')
fig_flash.savefig(FIG / 'flash.pdf', bbox_inches='tight')
plt.close(fig_flash)


## 6. Latência por configuração (média ± IC 95%)

In [9]:
kem = df[df.operation != 'msgcrypt'].copy()
s = summ[summ.operation != 'msgcrypt'].copy()
op_map = {'keygen': 'KeyGen', 'encaps': 'Encaps', 'decaps': 'Decaps'}
alg_map = {'kyber512': 'Kyber-512', 'ctru-light': 'CTRU-Light', 'ecdh': 'ECDH'}
panel_map = {
    ('asm', '-O3'): 'Assembly (-O3)',
    ('asm', '-Os'): 'Assembly (-Os)',
    ('ref', '-O3'): 'Referência (-O3)',
    ('ref', '-Os'): 'Referência (-Os)',
}
s['op_label'] = pd.Categorical(s.operation.map(op_map), categories=['KeyGen', 'Encaps', 'Decaps'], ordered=True)
s['alg_label'] = pd.Categorical(s.algorithm.map(alg_map), categories=['Kyber-512', 'CTRU-Light', 'ECDH'], ordered=True)
s['panel'] = pd.Categorical(
    [panel_map[(v, o)] for v, o in zip(s.variant, s.optimization)],
    categories=['Assembly (-O3)', 'Assembly (-Os)', 'Referência (-O3)', 'Referência (-Os)'],
    ordered=True
)
s['ymin'] = s.mean_ms - s.ci95_ms
s['ymax'] = s.mean_ms + s.ci95_ms
dodge = p9.position_dodge(width=0.35)
p_lat = (
    p9.ggplot(s, p9.aes(x='op_label', y='mean_ms', color='alg_label', group='alg_label'))
    + p9.geom_line(position=dodge, linetype='dashed', size=0.6, alpha=0.85)
    + p9.geom_errorbar(p9.aes(ymin='ymin', ymax='ymax'), position=dodge, width=0.18, size=0.5)
    + p9.geom_point(p9.aes(shape='alg_label'), position=dodge, size=2.2)
    + p9.facet_wrap('~ panel', nrow=1)
    + p9.scale_y_log10(breaks=[100, 1000, 10000], labels=['100 ms', '1 s', '10 s'], limits=(60, 25000))
    + p9.scale_color_manual(values={'Kyber-512': '#0072B2', 'CTRU-Light': '#D55E00', 'ECDH': '#009E73'}, name=' ')
    + p9.scale_shape_manual(values={'Kyber-512': 'o', 'CTRU-Light': 's', 'ECDH': '^'}, name=' ')
    + p9.labs(x='', y='Latência (escala log)')
    + p9.theme_bw(base_size=8)
    + p9.theme(
        legend_position='top',
        legend_direction='horizontal',
        legend_box_spacing=0.05,
        legend_margin=0,
        axis_text_x=p9.element_text(rotation=25, ha='right', size=7),
        strip_background=p9.element_rect(fill='#f0f0f0', color='#cccccc', size=0.5),
        strip_text=p9.element_text(weight='bold', size=7.5),
        panel_grid_minor=p9.element_blank(),
        panel_grid_major_x=p9.element_blank(),
        panel_grid_major_y=p9.element_line(color='#e0e0e0', size=0.4, linetype='dotted'),
        figure_size=(TEXTW, 2.5)
    )
)
fig_lat = p_lat.draw()
fig_lat.savefig(FIG / 'latency.png', dpi=300, bbox_inches='tight')
fig_lat.savefig(FIG / 'latency.pdf', bbox_inches='tight')
plt.close(fig_lat)


## 7. Comparação de alternativas (Welch, razão de médias em escala log)

In [10]:
def compare(a, b):
    # Razão das médias geométricas b/a com IC 95% (Welch sobre log).
    la, lb = np.log(a), np.log(b)
    d = lb.mean() - la.mean()
    se = np.sqrt(la.var(ddof=1) / len(la) + lb.var(ddof=1) / len(lb))
    dof = se**4 / ((la.var(ddof=1) / len(la))**2 / (len(la) - 1) + (lb.var(ddof=1) / len(lb))**2 / (len(lb) - 1))
    h = stats.t.ppf(0.975, dof) * se
    p = stats.ttest_ind(lb, la, equal_var=False).pvalue
    return np.exp(d), np.exp(d - h), np.exp(d + h), p

def table(factor, base, alt, fixed):
    out = []
    for keys, grp in kem.groupby([k for k in KEYS if k != factor]):
        a = grp.loc[grp[factor] == base, "t_exec_ms"]; b = grp.loc[grp[factor] == alt, "t_exec_ms"]
        if len(a) > 1 and len(b) > 1:
            r, lo, hi, p = compare(a, b)
            out.append({**dict(zip([k for k in KEYS if k != factor], keys)),
                        f"razao_{alt}/{base}": r, "ic_lo": lo, "ic_hi": hi, "p_welch": p})
    return pd.DataFrame(out)

ref_vs_asm = table("variant", "asm", "ref", None)
os_vs_o3 = table("optimization", "-O3", "-Os", None)
kyb_vs_ctru = table("algorithm", "ctru-light", "kyber512", None)
ecdh_vs_kyb = table("algorithm", "kyber512", "ecdh", None)
ecdh_vs_ctru = table("algorithm", "ctru-light", "ecdh", None)
for name, t in [("ref/asm", ref_vs_asm), ("-Os/-O3", os_vs_o3), ("kyber/ctru", kyb_vs_ctru),
                ("ecdh/kyber", ecdh_vs_kyb), ("ecdh/ctru", ecdh_vs_ctru)]:
    t.to_csv(OUT / f"cmp_{name.replace('/', '_vs_').replace('-', '')}.csv", index=False)
    print("\n==", name); display(t.round(4))


== ref/asm


,algorithm,optimization,operation,razao_ref/asm,ic_lo,ic_hi,p_welch
0,ctru-light,-O3,decaps,1.8050,1.8040,1.8060,0.0
1,ctru-light,-O3,encaps,1.5332,1.5095,1.5572,0.0
2,ctru-light,-O3,keygen,2.1364,2.0998,2.1736,0.0
3,ctru-light,-Os,decaps,1.8127,1.8062,1.8193,0.0
4,ctru-light,-Os,encaps,1.5494,1.5239,1.5752,0.0
5,ctru-light,-Os,keygen,2.1380,2.0996,2.1770,0.0
6,ecdh,-O3,decaps,2.1307,2.1302,2.1311,0.0
7,ecdh,-O3,encaps,2.1360,2.1356,2.1363,0.0
8,ecdh,-O3,keygen,2.1330,2.1326,2.1334,0.0
9,ecdh,-Os,decaps,2.2572,2.2566,2.2577,0.0



== -Os/-O3


,algorithm,variant,operation,razao_-Os/-O3,ic_lo,ic_hi,p_welch
0,ctru-light,asm,decaps,1.0368,1.0330,1.0406,0.0000
1,ctru-light,asm,encaps,1.0267,1.0097,1.0440,0.0023
2,ctru-light,asm,keygen,1.0264,1.0053,1.0480,0.0144
3,ctru-light,ref,decaps,1.0413,1.0407,1.0418,0.0000
4,ctru-light,ref,encaps,1.0375,1.0217,1.0536,0.0000
5,ctru-light,ref,keygen,1.0272,1.0130,1.0415,0.0003
6,ecdh,asm,decaps,1.0060,1.0057,1.0062,0.0000
7,ecdh,asm,encaps,1.0056,1.0054,1.0058,0.0000
8,ecdh,asm,keygen,1.0057,1.0055,1.0059,0.0000
9,ecdh,ref,decaps,1.0657,1.0655,1.0659,0.0000



== kyber/ctru


,variant,optimization,operation,razao_kyber512/ctru-light,ic_lo,ic_hi,p_welch
0,asm,-O3,decaps,2.0976,2.0958,2.0994,0.0
1,asm,-O3,encaps,3.7668,3.7235,3.8106,0.0
2,asm,-O3,keygen,2.8752,2.8337,2.9173,0.0
3,asm,-Os,decaps,2.1323,2.1244,2.1402,0.0
4,asm,-Os,encaps,3.8666,3.8192,3.9146,0.0
5,asm,-Os,keygen,2.9326,2.8883,2.9776,0.0
6,ref,-O3,decaps,2.2766,2.2746,2.2786,0.0
7,ref,-O3,encaps,4.4972,4.4491,4.5459,0.0
8,ref,-O3,keygen,2.5622,2.5370,2.5876,0.0
9,ref,-Os,decaps,2.2354,2.2333,2.2375,0.0



== ecdh/kyber


,variant,optimization,operation,razao_ecdh/kyber512,ic_lo,ic_hi,p_welch
0,asm,-O3,decaps,10.3465,10.3388,10.3541,0.0
1,asm,-O3,encaps,18.9997,18.9875,19.0119,0.0
2,asm,-O3,keygen,12.5030,12.4895,12.5166,0.0
3,asm,-Os,decaps,9.8752,9.8668,9.8835,0.0
4,asm,-Os,encaps,18.1291,18.1115,18.1467,0.0
5,asm,-Os,keygen,12.0111,12.0001,12.0221,0.0
6,ref,-O3,decaps,11.2531,11.2440,11.2621,0.0
7,ref,-O3,encaps,22.1705,22.1544,22.1866,0.0
8,ref,-O3,keygen,14.0079,13.9950,14.0207,0.0
9,ref,-Os,decaps,11.7292,11.7189,11.7395,0.0



== ecdh/ctru


,variant,optimization,operation,razao_ecdh/ctru-light,ic_lo,ic_hi,p_welch
0,asm,-O3,decaps,21.7029,21.6928,21.7131,0.0
1,asm,-O3,encaps,71.5679,70.7459,72.3994,0.0
2,asm,-O3,keygen,35.9487,35.4317,36.4733,0.0
3,asm,-Os,decaps,21.0570,20.9809,21.1335,0.0
4,asm,-Os,encaps,70.0983,69.2416,70.9656,0.0
5,asm,-Os,keygen,35.2238,34.6919,35.7638,0.0
6,ref,-O3,decaps,25.6185,25.6079,25.6291,0.0
7,ref,-O3,encaps,99.7060,98.6407,100.7828,0.0
8,ref,-O3,keygen,35.8905,35.5394,36.2451,0.0
9,ref,-Os,decaps,26.2200,26.2091,26.2308,0.0


## 8. Modelo linear (ANOVA fatorial sobre log da latência)

Modelo: $\log(T) = \mu + \text{alg} + \text{variante} + \text{opt} + \text{op} + \text{interações} + \varepsilon$,
somente KEM (keygen/encaps/decaps; `msgcrypt` é um XOR com o segredo, fora do KEM).
O log transforma efeitos multiplicativos (speedup) em aditivos e estabiliza a
variância entre configurações com latências muito diferentes. A fração da
soma de quadrados explicada por cada termo ($\eta^2$) mostra quais fatores
dominam o desempenho.

In [11]:
kem["logT"] = np.log(kem.t_exec_ms)
fit = smf.ols("logT ~ C(algorithm) * C(variant) * C(optimization) * C(operation)", data=kem).fit()
aov = sm.stats.anova_lm(fit, typ=2)
aov["eta2"] = aov.sum_sq / aov.sum_sq.sum()
aov = aov.sort_values("eta2", ascending=False)
aov.to_csv(OUT / "anova_latency.csv")
print(f"R² = {fit.rsquared:.5f}   n = {int(fit.nobs)}")
aov.round(5)

R² = 0.99964   n = 2160


,sum_sq,df,F,PR(>F),eta2
C(algorithm),1851.63667,2.0,2.504751e+06,0.00000,0.86707
C(variant),210.25782,1.0,5.688410e+05,0.00000,0.09846
C(algorithm):C(operation),43.25963,4.0,2.925914e+04,0.00000,0.02026
C(operation),25.47052,2.0,3.445456e+04,0.00000,0.01193
C(algorithm):C(variant):C(operation),1.31584,4.0,8.899818e+02,0.00000,0.00062
C(algorithm):C(variant),1.11882,2.0,1.513450e+03,0.00000,0.00052
C(variant):C(operation),1.05162,2.0,1.422551e+03,0.00000,0.00049
Residual,0.78508,2124.0,NaN,NaN,0.00037
C(optimization),0.49985,1.0,1.352320e+03,0.00000,0.00023
C(algorithm):C(variant):C(optimization),0.07787,2.0,1.053340e+02,0.00000,0.00004


In [12]:
# Modelo aditivo (sem interações): coeficientes interpretáveis como fatores multiplicativos.
add = smf.ols("logT ~ C(algorithm) + C(variant) + C(optimization) + C(operation)", data=kem).fit()
coef = pd.DataFrame({"fator_multiplicativo": np.exp(add.params),
                     "ic_lo": np.exp(add.conf_int()[0]), "ic_hi": np.exp(add.conf_int()[1])})
print(f"R² aditivo = {add.rsquared:.4f}")
coef.round(4)

R² aditivo = 0.9779


,fator_multiplicativo,ic_lo,ic_hi
Intercept,117.5290,115.5057,119.5879
C(algorithm)[T.ecdh],40.2613,39.2627,41.2852
C(algorithm)[T.kyber512],2.8844,2.8407,2.9287
C(variant)[T.ref],1.8890,1.8649,1.9133
C(optimization)[T.-Os],1.0339,1.0197,1.0482
C(operation)[T.encaps],0.9495,0.9350,0.9642
C(operation)[T.keygen],0.7774,0.7655,0.7894


In [13]:
fig, axs = plt.subplots(1, 2, figsize=(7, 2.6))
axs[0].scatter(fit.fittedvalues, fit.resid, s=3, alpha=.4); axs[0].axhline(0, c="k", lw=.8)
axs[0].set_xlabel("ajustado (log ms)"); axs[0].set_ylabel("resíduo")
sm.qqplot(fit.resid, line="s", ax=axs[1], markersize=2); axs[1].set_title("")
fig.savefig(FIG / "residuals.pdf"); fig.savefig(FIG / "residuals.png", dpi=200)

## 9. Trade-off memória × latência

In [14]:
s = summ[summ.operation != 'msgcrypt'].copy()
alg_map = {'kyber512': 'Kyber-512', 'ctru-light': 'CTRU-Light', 'ecdh': 'ECDH'}
s['cfg_label'] = [f"{alg_map[a]} ({v})" for a, v in zip(s.algorithm, s.variant)]
cfg_order = ['Kyber-512 (asm)', 'Kyber-512 (ref)', 'CTRU-Light (asm)', 'CTRU-Light (ref)', 'ECDH (asm)', 'ECDH (ref)']
s['cfg_label'] = pd.Categorical(s.cfg_label, categories=cfg_order, ordered=True)
s['ymin'] = s.mean_ms - s.ci95_ms
s['ymax'] = s.mean_ms + s.ci95_ms
colors = {
    'Kyber-512 (asm)': '#0072B2', 'Kyber-512 (ref)': '#0072B2',
    'CTRU-Light (asm)': '#D55E00', 'CTRU-Light (ref)': '#D55E00',
    'ECDH (asm)': '#009E73', 'ECDH (ref)': '#009E73'
}
fills = {
    'Kyber-512 (asm)': '#0072B2', 'Kyber-512 (ref)': 'white',
    'CTRU-Light (asm)': '#D55E00', 'CTRU-Light (ref)': 'white',
    'ECDH (asm)': '#009E73', 'ECDH (ref)': 'white'
}
shapes = {
    'Kyber-512 (asm)': 'o', 'Kyber-512 (ref)': 's',
    'CTRU-Light (asm)': 'o', 'CTRU-Light (ref)': 's',
    'ECDH (asm)': 'o', 'ECDH (ref)': 's'
}
p_trade = (
    p9.ggplot(s, p9.aes(x='stack_peak_b', y='mean_ms', color='cfg_label', fill='cfg_label', shape='cfg_label'))
    + p9.geom_errorbar(p9.aes(ymin='ymin', ymax='ymax'), width=25, size=0.5, alpha=0.8)
    + p9.geom_point(size=2.8, stroke=1.1)
    + p9.scale_x_continuous(
        limits=(350, 3500),
        breaks=[500, 1000, 1500, 2000, 2500, 3000],
        labels=['500 B', '1.000 B', '1.500 B', '2.000 B', '2.500 B', '3.000 B']
    )
    + p9.scale_y_log10(limits=(60, 25000), breaks=[100, 1000, 10000], labels=['100 ms', '1 s', '10 s'])
    + p9.scale_color_manual(values=colors, name='Configuração')
    + p9.scale_fill_manual(values=fills, name='Configuração')
    + p9.scale_shape_manual(values=shapes, name='Configuração')
    + p9.guides(color=p9.guide_legend(ncol=2), fill=p9.guide_legend(ncol=2), shape=p9.guide_legend(ncol=2))
    + p9.labs(x='Pico de pilha (bytes)', y='Latência média (escala log)')
    + p9.theme_bw(base_size=8)
    + p9.theme(
        legend_position=(0.77, 0.77),
        legend_background=p9.element_rect(fill='white', color='#d0d0d0', size=0.5),
        legend_title=p9.element_blank(),
        legend_text=p9.element_text(size=7),
        legend_key=p9.element_blank(),
        legend_box_margin=2,
        panel_grid_minor=p9.element_blank(),
        panel_grid_major=p9.element_line(color='#e5e5e5', size=0.4, linetype='dotted'),
        figure_size=(TEXTW, 2.5)
    )
)
fig_trade = p_trade.draw()
ax_tr = fig_trade.axes[0]
ax_top = ax_tr.twiny()
ax_top.set_xlim(ax_tr.get_xlim())
top_pcts = [10, 20, 30, 40, 50]
top_ticks = [6144 * p_val / 100 for p_val in top_pcts]
ax_top.set_xticks(top_ticks)
ax_top.set_xticklabels([f"{p_val}%" for p_val in top_pcts], fontsize=7)
ax_top.set_xlabel('Ocupação da SRAM do ATmega4809 (total 6 KB)', fontsize=7.5, labelpad=4)
ax_top.grid(False)
fig_trade.savefig(FIG / 'tradeoff.png', dpi=300, bbox_inches='tight')
fig_trade.savefig(FIG / 'tradeoff.pdf', bbox_inches='tight')
plt.close(fig_trade)


## 9b. Tabelas exportadas para o artigo (`artigo/tables/`)

In [15]:
TAB = ROOT / "artigo" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
t = summ[summ.operation != "msgcrypt"].set_index(["algorithm", "variant", "optimization", "operation"])
fl = flash.set_index(["algorithm", "variant", "optimization"]).flash_b
lines = [r"\begin{tabular}{@{}lllrrrrrr@{}}", r"\toprule",
         r" & & & \multicolumn{3}{c}{Latência (ms), média $\pm$ IC 95\%} & & & \\",
         r"\cmidrule(lr){4-6}",
         r"Algoritmo & Var. & Otim. & KeyGen & Encaps & Decaps & Pilha (B) & Flash (KB) & $N$ \\",
         r"\midrule"]
for k, alg in enumerate(["kyber512", "ctru-light", "ecdh"]):
    if k: lines.append(r"\addlinespace")
    first = True
    for var in ["asm", "ref"]:
        for opt in ["-O3", "-Os"]:
            r = t.loc[(alg, var, opt)]
            cells = [f"{br(r.loc[o, 'mean_ms'])} $\\pm$ {br(r.loc[o, 'ci95_ms'])}" for o in ["keygen", "encaps", "decaps"]]
            lines.append(" & ".join([LABEL[alg] if first else "", var, f"$-${opt[1:]}", *cells,
                                     f"{int(r.stack_peak_b.max()):,}".replace(",", "."),
                                     br(fl.loc[(alg, var, opt)] / 1024),
                                     str(int(r.n.min()))]) + r" \\")
            first = False
lines += [r"\bottomrule", r"\end{tabular}"]
(TAB / "results.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")

NOME = {"algorithm": "Algoritmo", "variant": "Implementação", "optimization": "Otimização",
        "operation": "Operação"}
def termo(i):
    return r" $\times$ ".join(NOME[f.strip()[2:-1]] for f in i.split(":"))
top = aov.drop(index="Residual").head(6)
lines = [r"\begin{tabular}{@{}lrrr@{}}", r"\toprule",
         r"Termo & GL & $F$ & $\eta^2$ (\%) \\", r"\midrule"]
for i, r in top.iterrows():
    lines.append(f"{termo(i)} & {int(r.df)} & {br(r.F, 0)} & {br(100 * r.eta2, 2)} \\\\")
r = aov.loc["Residual"]
lines += [r"\midrule", f"Resíduo & {int(r.df)} & -- & {br(100 * r.eta2, 2)} \\\\",
          r"\bottomrule", r"\end{tabular}"]
(TAB / "anova.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
print((TAB / "results.tex").read_text(encoding="utf-8")); print((TAB / "anova.tex").read_text(encoding="utf-8"))

\begin{tabular}{@{}lllrrrrrr@{}}
\toprule
 & & & \multicolumn{3}{c}{Latência (ms), média $\pm$ IC 95\%} & & & \\
\cmidrule(lr){4-6}
Algoritmo & Var. & Otim. & KeyGen & Encaps & Decaps & Pilha (B) & Flash (KB) & $N$ \\
\midrule
Kyber-512 & asm & $-$O3 & 261,8 $\pm$ 0,3 & 343,7 $\pm$ 0,2 & 317,0 $\pm$ 0,2 & 1.019 & 25,6 & 185 \\
 & asm & $-$Os & 274,1 $\pm$ 0,3 & 362,2 $\pm$ 0,4 & 334,1 $\pm$ 0,3 & 997 & 15,1 & 60 \\
 & ref & $-$O3 & 498,4 $\pm$ 0,5 & 629,1 $\pm$ 0,5 & 621,1 $\pm$ 0,5 & 976 & 46,1 & 175 \\
 & ref & $-$Os & 508,7 $\pm$ 0,6 & 642,0 $\pm$ 0,6 & 635,0 $\pm$ 0,6 & 939 & 35,6 & 60 \\
\addlinespace
CTRU-Light & asm & $-$O3 & 91,2 $\pm$ 1,3 & 91,3 $\pm$ 1,1 & 151,1 $\pm$ 0,1 & 3.177 & 28,1 & 60 \\
 & asm & $-$Os & 93,6 $\pm$ 1,4 & 93,8 $\pm$ 1,2 & 156,7 $\pm$ 0,6 & 3.137 & 22,6 & 60 \\
 & ref & $-$O3 & 194,6 $\pm$ 1,9 & 139,9 $\pm$ 1,5 & 272,8 $\pm$ 0,1 & 3.049 & 46,0 & 30 \\
 & ref & $-$Os & 199,9 $\pm$ 2,0 & 145,2 $\pm$ 1,7 & 284,1 $\pm$ 0,1 & 3.009 & 40,0 & 30 \\
\addlinespac

## 10. Interpretação

*Texto escrito sobre a execução de 2026-11-05 com 10 campanhas de bancada
(2880 observações, 2160 de KEM, 20 MHz), já incluindo o ECDH. Se os dados
mudarem, revisar os números abaixo.*

**Quais fatores dominam a latência.** Na ANOVA sobre $\log T$, o algoritmo
explica ~87% da variação e a implementação (asm × ref) ~10%. A operação e a
interação algoritmo×operação explicam ~3% juntas, e a otimização do compilador
(-Os × -O3) < 0,1%. O modelo completo tem $R^2 > 0{,}999$. A fatia do
algoritmo cresceu em relação à análise só com PQC (~58%) porque o ECDH é uma
ordem de grandeza mais lento: ele domina a variação total.

**PQC × clássico.** No ATmega4809, o ECDH é **mais lento** que os dois KEMs
pós-quânticos: de 10× a 23× mais lento que o Kyber-512 e de 21× a 102× mais
lento que o CTRU-Light (por exemplo, Encaps asm -O3: 6,5 s contra 344 ms e
91 ms). É o oposto da intuição de que PQC é cara. Uma explicação provável (não
verificada por perfilamento): com multiplicador de só 8×8 bits, a aritmética
modular de 256 bits da curva custa mais que os polinômios de coeficientes
pequenos dos reticulados. A desvantagem do PQC está na memória:
o ECDH usa de 620 a 893 B de pilha e cabe em 9–15 KB de Flash (asm).

**Assembly AVR.** A implementação `asm` é 1,5×–2,3× mais rápida que `ref` em
todas as células. Ela também reduz muito a Flash: o Kyber -O3 cai de 46 KB
(só 4% livre) para 26 KB. Sem `asm`, o Kyber `ref -O3` praticamente não cabe
junto com uma aplicação.

**-Os × -O3.** -Os é de 0,6% a 7% mais lento, mas economiza até 28 KB de Flash
(ECDH ref: 37,8 → 10,5 KB). No ATmega4809, onde a Flash é o recurso escasso,
-Os é a escolha dominante.

**Kyber-512 × CTRU-Light.** O CTRU-Light é 2,1×–4,5× mais rápido (o maior
ganho é no Encaps), mas usa cerca de 3× mais pilha (2,3–3,2 KB contra
~1,0 KB), deixando 48–62% da SRAM livre contra 83–86%. É um *trade-off* real:
o CTRU-Light troca memória por tempo.

**Variabilidade.** O CV fica abaixo de 1% em todas as células, exceto
KeyGen/Encaps do CTRU-Light (CV de 3–6%, mediana > média), o que sugere uma
distribuição bimodal, provavelmente por amostragem com rejeição; ainda não
verificado no código. O ECDH é o mais estável (CV ≈ 0,03%). O `msgcrypt` (XOR
com o segredo, o mesmo código em todos os firmwares) fica entre 38,5 e
39,8 ms em -O3: serve de controle de estabilidade e mostra a ordem de grandeza
do overhead serial.

**Limitações.** (1) A latência inclui o overhead serial do host; as razões
entre configurações são menos afetadas que os valores absolutos (para o
ECDH, com segundos por operação, o overhead é desprezível). (2) Só 20 MHz foi
coletado. (3) Energia não medida (shunt pendente). (4) A ordem de execução
não foi aleatorizada na bancada. (5) Células desbalanceadas (N de 10 a 185).
(6) Os resíduos não são normais nas células bimodais; com N grande e efeitos
desta magnitude, as conclusões não dependem disso.